# 🎯 Ejercicios Resueltos — Clasificación

Este notebook resuelve los ejercicios propuestos al final de **`EDA_Clasificacion.ipynb`**.

### Ejercicios que abordaremos:

1. **Repite el análisis desactivando SMOTE** — ¿qué pasa con Precision y Recall?
2. **Cambia `max_depth` de RandomForest a 2 y a 30** — subajuste vs sobreajuste extremos.
3. **Añade regularización L2 al MLP de Keras** — ¿reduce el sobreajuste?
4. **Usa `class_weight='balanced'` como alternativa a SMOTE** — comparativa.
5. **Síntesis:** ¿cuál es la mejor combinación para este problema?

> 💡 Estructura de cada ejercicio: **Enunciado → Hipótesis → Implementación → Resultados → Conclusión**.


## Setup común

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 90
print("✅ Setup completo")

In [ ]:
# Cargamos y desbalanceamos como en el notebook base
data = load_breast_cancer(as_frame=True)
df = data.frame.rename(columns={'target': 'clase'})
df['clase'] = 1 - df['clase']  # 1 = maligno (positivo)

np.random.seed(SEED)
clase_0 = df[df['clase'] == 0]
clase_1 = df[df['clase'] == 1].sample(frac=0.20, random_state=SEED)
df = pd.concat([clase_0, clase_1]).sample(frac=1, random_state=SEED).reset_index(drop=True)

# Winsorización
for col in df.columns.drop('clase'):
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df[col] = df[col].clip(lower=Q1-1.5*IQR, upper=Q3+1.5*IQR)

X = df.drop(columns='clase').values
y = df['clase'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# SMOTE (aplicado a versiones raw y scaled del train)
X_train_bal,     y_train_bal     = SMOTE(random_state=SEED).fit_resample(X_train_sc, y_train)
X_train_bal_raw, y_train_bal_raw = SMOTE(random_state=SEED).fit_resample(X_train, y_train)

print(f"Balance train original: {dict(pd.Series(y_train).value_counts())}")
print(f"Balance train con SMOTE: {dict(pd.Series(y_train_bal).value_counts())}")

## Ejercicio 1 · ¿Qué pasa sin SMOTE?

### 📝 Enunciado
Entrenar los mismos modelos **sin** balanceo de clases y comparar contra la versión con SMOTE. Prestar especial atención al **Recall de la clase minoritaria** (maligno).

### 💭 Hipótesis
Sin SMOTE, los modelos aprenderán a "apostar" a la clase mayoritaria (benigno). Esperamos:
- Accuracy similar (o incluso mayor).
- **Recall de la clase maligna colapsará** — el modelo detectará menos casos malignos.
- La Precision podría subir (los pocos positivos que predice serán muy seguros).


In [ ]:
def evaluar(nombre, y_true, y_pred, y_proba):
    return {
        'Modelo': nombre,
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred),
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'AUC':       roc_auc_score(y_true, y_proba)
    }

def entrenar_modelo(nombre, X_tr, y_tr):
    if nombre == 'RF':
        m = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
    elif nombre == 'XGB':
        m = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                          eval_metric='logloss', random_state=SEED, n_jobs=-1)
    elif nombre == 'MLP':
        m = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=200, random_state=SEED,
                          early_stopping=True, validation_fraction=0.15)
    m.fit(X_tr, y_tr)
    return m

resultados_smote = []
predicciones_smote = {}

for nombre in ['RF', 'XGB', 'MLP']:
    # SIN SMOTE
    if nombre == 'MLP':
        m_sin = entrenar_modelo(nombre, X_train_sc, y_train)
        pred_sin  = m_sin.predict(X_test_sc)
        proba_sin = m_sin.predict_proba(X_test_sc)[:,1]
    else:
        m_sin = entrenar_modelo(nombre, X_train, y_train)
        pred_sin  = m_sin.predict(X_test)
        proba_sin = m_sin.predict_proba(X_test)[:,1]
    res = evaluar(f'{nombre} (SIN SMOTE)', y_test, pred_sin, proba_sin)
    resultados_smote.append(res)
    predicciones_smote[f'{nombre} sin'] = pred_sin

    # CON SMOTE
    if nombre == 'MLP':
        m_con = entrenar_modelo(nombre, X_train_bal, y_train_bal)
        pred_con  = m_con.predict(X_test_sc)
        proba_con = m_con.predict_proba(X_test_sc)[:,1]
    else:
        m_con = entrenar_modelo(nombre, X_train_bal_raw, y_train_bal_raw)
        pred_con  = m_con.predict(X_test)
        proba_con = m_con.predict_proba(X_test)[:,1]
    res = evaluar(f'{nombre} (CON SMOTE)', y_test, pred_con, proba_con)
    resultados_smote.append(res)
    predicciones_smote[f'{nombre} con'] = pred_con

df_smote = pd.DataFrame(resultados_smote).set_index('Modelo')
df_smote.round(4)

In [ ]:
# Visualización comparativa: Precision y Recall
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

modelos = ['RF', 'XGB', 'MLP']
x = np.arange(len(modelos))
w = 0.35

recall_sin = [df_smote.loc[f'{m} (SIN SMOTE)', 'Recall'] for m in modelos]
recall_con = [df_smote.loc[f'{m} (CON SMOTE)', 'Recall'] for m in modelos]
precision_sin = [df_smote.loc[f'{m} (SIN SMOTE)', 'Precision'] for m in modelos]
precision_con = [df_smote.loc[f'{m} (CON SMOTE)', 'Precision'] for m in modelos]

axes[0].bar(x - w/2, recall_sin, w, label='SIN SMOTE', color='salmon')
axes[0].bar(x + w/2, recall_con, w, label='CON SMOTE', color='steelblue')
axes[0].set_xticks(x); axes[0].set_xticklabels(modelos)
axes[0].set_title('Recall (¿cuántos malignos detectamos?)'); axes[0].set_ylim(0, 1.05)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x - w/2, precision_sin, w, label='SIN SMOTE', color='salmon')
axes[1].bar(x + w/2, precision_con, w, label='CON SMOTE', color='steelblue')
axes[1].set_xticks(x); axes[1].set_xticklabels(modelos)
axes[1].set_title('Precision (¿de los que predije, cuántos aciertan?)'); axes[1].set_ylim(0, 1.05)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Matrices de confusión SIN vs CON SMOTE para el RF
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (etiqueta, pred) in zip(axes, [('SIN SMOTE', predicciones_smote['RF sin']),
                                        ('CON SMOTE', predicciones_smote['RF con'])]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Benigno', 'Maligno'],
                yticklabels=['Benigno', 'Maligno'], cbar=False)
    ax.set_title(f'Random Forest — {etiqueta}')
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
plt.tight_layout(); plt.show()

### ✅ Conclusión del Ejercicio 1

- **Sin SMOTE**, el Recall cae significativamente en todos los modelos: dejan pasar casos malignos sin detectar (falsos negativos ↑).
- **Con SMOTE**, la Precision puede bajar un poco (algún falso positivo extra), pero el Recall sube — se detectan más casos malignos.
- **En contexto médico**, un falso negativo (no detectar un tumor maligno) es catastrófico. Preferimos maximizar Recall aunque bajemos algo de Precision.
- La Accuracy sola es **engañosa** — con clases desbalanceadas puede quedarse alta mientras el Recall de la minoritaria colapsa.


## Ejercicio 2 · Impacto de `max_depth` en Random Forest

### 📝 Enunciado
Entrenar Random Forest con `max_depth ∈ {2, 5, 10, 30, None}` y observar la evolución de Accuracy, AUC y la brecha train-test.

### 💭 Hipótesis
- `max_depth=2`: subajuste — árboles poco profundos no capturan patrones complejos.
- `max_depth=30` o `None`: sobreajuste — cada árbol memoriza casi el train.
- Habrá un **valor óptimo intermedio**.


In [ ]:
profundidades = [2, 5, 10, 15, 30, None]
resultados_depth = []

for d in profundidades:
    rf = RandomForestClassifier(n_estimators=200, max_depth=d, random_state=SEED, n_jobs=-1)
    rf.fit(X_train_bal_raw, y_train_bal_raw)
    
    acc_tr = accuracy_score(y_train, rf.predict(X_train))
    acc_te = accuracy_score(y_test,  rf.predict(X_test))
    auc_te = roc_auc_score(y_test,   rf.predict_proba(X_test)[:,1])
    rec_te = recall_score(y_test,    rf.predict(X_test))
    
    resultados_depth.append({
        'max_depth': str(d),
        'Acc_train': acc_tr,
        'Acc_test':  acc_te,
        'Recall_test': rec_te,
        'AUC_test':  auc_te,
        'Gap Acc':   acc_tr - acc_te
    })

df_depth = pd.DataFrame(resultados_depth)
df_depth.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
x = np.arange(len(df_depth))

axes[0].plot(x, df_depth['Acc_train'], 'o-', label='Acc Train', color='steelblue', linewidth=2, markersize=9)
axes[0].plot(x, df_depth['Acc_test'],  'o-', label='Acc Test',  color='salmon',    linewidth=2, markersize=9)
axes[0].plot(x, df_depth['AUC_test'],  's--',label='AUC Test',  color='green',     linewidth=2, markersize=9)
axes[0].set_xticks(x); axes[0].set_xticklabels(df_depth['max_depth'])
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('Métrica')
axes[0].set_title('Métricas según profundidad'); axes[0].legend(); axes[0].grid(True)

colores = ['lightgreen' if g < 0.05 else 'gold' if g < 0.10 else 'crimson' for g in df_depth['Gap Acc']]
axes[1].bar(x, df_depth['Gap Acc'], color=colores)
axes[1].axhline(0.10, color='crimson', linestyle='--', alpha=0.5, label='Umbral sobreajuste')
axes[1].set_xticks(x); axes[1].set_xticklabels(df_depth['max_depth'])
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Gap Accuracy (train − test)')
axes[1].set_title('Brecha train − test'); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

### ✅ Conclusión del Ejercicio 2

- Con `max_depth=2`: modelo demasiado simple, pero en este dataset (pocas features altamente predictivas) el subajuste es menos dramático de lo que uno esperaría.
- Con `max_depth ≥ 10`: la accuracy en train se acerca a 1.0 mientras la de test se estanca → **sobreajuste**.
- El AUC es más estable que la accuracy y muestra que a partir de `max_depth ≈ 5-10` el modelo ya no mejora.
- **Regla práctica:** empieza con `max_depth` bajo y sube hasta encontrar la profundidad donde el Gap empieza a crecer significativamente.


## Ejercicio 3 · Regularización L2 en el MLP de Keras

### 📝 Enunciado
Comparar tres redes Keras idénticas en arquitectura pero con **regularizaciones distintas**:
- Sin regularización.
- Con Dropout (como en el notebook base).
- Con L2 (penalización de pesos grandes).

### 💭 Hipótesis
Las versiones regularizadas mostrarán menor gap train-val → menos sobreajuste, pero podrían necesitar más épocas para converger.


In [ ]:
def crear_modelo(tipo):
    model = Sequential()
    if tipo == 'sin_reg':
        model.add(Dense(64, activation='relu', input_shape=(X_train_bal.shape[1],)))
        model.add(Dense(32, activation='relu'))
    elif tipo == 'dropout':
        model.add(Dense(64, activation='relu', input_shape=(X_train_bal.shape[1],)))
        model.add(Dropout(0.3))
        model.add(Dense(32, activation='relu'))
        model.add(Dropout(0.3))
    elif tipo == 'l2':
        model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.01),
                        input_shape=(X_train_bal.shape[1],)))
        model.add(Dense(32, activation='relu', kernel_regularizer=l2(0.01)))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

tipos = ['sin_reg', 'dropout', 'l2']
historias = {}
resultados_reg = []

for tipo in tipos:
    model = crear_modelo(tipo)
    early = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    hist = model.fit(X_train_bal, y_train_bal, validation_split=0.2,
                     epochs=80, batch_size=32, callbacks=[early], verbose=0)
    historias[tipo] = hist.history
    
    y_te_proba = model.predict(X_test_sc, verbose=0).ravel()
    y_te_pred  = (y_te_proba > 0.5).astype(int)
    y_tr_proba = model.predict(X_train_sc, verbose=0).ravel()
    y_tr_pred  = (y_tr_proba > 0.5).astype(int)
    
    resultados_reg.append({
        'Regularización': tipo,
        'Acc_train': accuracy_score(y_train, y_tr_pred),
        'Acc_test':  accuracy_score(y_test,  y_te_pred),
        'AUC_test':  roc_auc_score(y_test,   y_te_proba),
        'Recall_test': recall_score(y_test,  y_te_pred),
        'Épocas':    len(hist.history['loss'])
    })

df_reg = pd.DataFrame(resultados_reg)
df_reg['Gap'] = df_reg['Acc_train'] - df_reg['Acc_test']
df_reg.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
etiquetas = {'sin_reg': 'Sin regularización', 'dropout': 'Con Dropout', 'l2': 'Con L2'}
colores_tipo = {'sin_reg': 'crimson', 'dropout': 'steelblue', 'l2': 'green'}

for tipo in tipos:
    h = historias[tipo]
    axes[0].plot(h['loss'],     '--', color=colores_tipo[tipo], alpha=0.6, label=f'{etiquetas[tipo]} — Train')
    axes[0].plot(h['val_loss'], '-',  color=colores_tipo[tipo], label=f'{etiquetas[tipo]} — Val')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].set_title('Curvas de pérdida (train vs val)'); axes[0].legend(fontsize=8); axes[0].grid(True)

for tipo in tipos:
    h = historias[tipo]
    axes[1].plot(h['accuracy'],     '--', color=colores_tipo[tipo], alpha=0.6, label=f'{etiquetas[tipo]} — Train')
    axes[1].plot(h['val_accuracy'], '-',  color=colores_tipo[tipo], label=f'{etiquetas[tipo]} — Val')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Curvas de accuracy'); axes[1].legend(fontsize=8); axes[1].grid(True)

# Comparativa del gap
axes[2].bar(range(len(df_reg)), df_reg['Gap'],
            color=[colores_tipo[t] for t in df_reg['Regularización']])
axes[2].set_xticks(range(len(df_reg)))
axes[2].set_xticklabels([etiquetas[t] for t in df_reg['Regularización']], rotation=15)
axes[2].set_ylabel('Gap Acc (train − test)')
axes[2].set_title('Sobreajuste según regularización'); axes[2].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

### ✅ Conclusión del Ejercicio 3

- **Sin regularización**: la red aprende rápido pero el gap train-val se ensancha → sobreajuste claro.
- **Con Dropout**: el gap se reduce; la curva de train es más ruidosa (por el ruido inyectado en cada mini-batch).
- **Con L2**: el gap también baja; la curva train es más suave que con Dropout.
- **En la práctica**, se pueden combinar Dropout + L2. El valor `l2(0.01)` es un buen punto de partida; se puede ajustar con validación cruzada.
- Ninguna técnica es universalmente mejor; depende del dataset y arquitectura.


## Ejercicio 4 · `class_weight='balanced'` como alternativa a SMOTE

### 📝 Enunciado
En lugar de generar muestras sintéticas (SMOTE), podemos **penalizar más los errores** en la clase minoritaria durante el entrenamiento (`class_weight='balanced'`). Comparamos las tres estrategias:
1. Baseline: sin balanceo.
2. SMOTE (oversampling sintético).
3. `class_weight='balanced'` (ponderación).

### 💭 Hipótesis
Ambas técnicas de balanceo mejorarán el Recall vs baseline. `class_weight` suele ser más rápido porque no aumenta el tamaño del train.


In [ ]:
def entrenar_estrategia(nombre_modelo, estrategia):
    if nombre_modelo == 'RF':
        params = {'n_estimators': 200, 'max_depth': 10, 'random_state': SEED, 'n_jobs': -1}
        if estrategia == 'class_weight':
            params['class_weight'] = 'balanced'
        m = RandomForestClassifier(**params)
    else:  # XGB
        # XGBoost no tiene class_weight directo, pero usa scale_pos_weight
        neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
        params = {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05,
                  'eval_metric': 'logloss', 'random_state': SEED, 'n_jobs': -1}
        if estrategia == 'class_weight':
            params['scale_pos_weight'] = neg / pos
        m = XGBClassifier(**params)
    
    if estrategia == 'smote':
        m.fit(X_train_bal_raw, y_train_bal_raw)
    else:
        m.fit(X_train, y_train)
    
    return m

resultados_cw = []
tiempos = {}

for nombre_modelo in ['RF', 'XGB']:
    for estrategia in ['baseline', 'smote', 'class_weight']:
        t0 = time.time()
        m = entrenar_estrategia(nombre_modelo, estrategia)
        t_ent = time.time() - t0
        
        y_pred  = m.predict(X_test)
        y_proba = m.predict_proba(X_test)[:, 1]
        
        resultados_cw.append({
            'Modelo': nombre_modelo,
            'Estrategia': estrategia,
            'Accuracy':  accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall':    recall_score(y_test, y_pred),
            'F1':        f1_score(y_test, y_pred, zero_division=0),
            'AUC':       roc_auc_score(y_test, y_proba),
            'Tiempo (s)': round(t_ent, 2)
        })

df_cw = pd.DataFrame(resultados_cw)
df_cw.round(4)

In [ ]:
# Visualización: Precision-Recall por estrategia
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metricas_plot = ['Precision', 'Recall', 'F1', 'AUC']
estrategias = ['baseline', 'smote', 'class_weight']
colores_est = {'baseline': 'salmon', 'smote': 'steelblue', 'class_weight': 'green'}

for ax, modelo in zip(axes, ['RF', 'XGB']):
    sub = df_cw[df_cw['Modelo'] == modelo].set_index('Estrategia')
    x = np.arange(len(metricas_plot))
    w = 0.25
    for i, est in enumerate(estrategias):
        ax.bar(x + i*w, [sub.loc[est, m] for m in metricas_plot], w,
               label=est, color=colores_est[est])
    ax.set_xticks(x + w)
    ax.set_xticklabels(metricas_plot)
    ax.set_ylim(0, 1.05); ax.set_title(f'{modelo}: comparativa de estrategias')
    ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Comparación de tiempos: SMOTE vs class_weight
plt.figure(figsize=(9, 4))
sub_rf = df_cw[df_cw['Modelo']=='RF'].set_index('Estrategia')
sub_xgb = df_cw[df_cw['Modelo']=='XGB'].set_index('Estrategia')

x = np.arange(len(estrategias))
w = 0.35
plt.bar(x - w/2, [sub_rf.loc[e, 'Tiempo (s)']  for e in estrategias], w, label='RF',  color='steelblue')
plt.bar(x + w/2, [sub_xgb.loc[e, 'Tiempo (s)'] for e in estrategias], w, label='XGB', color='salmon')
plt.xticks(x, estrategias)
plt.ylabel('Tiempo de entrenamiento (s)')
plt.title('Costo computacional por estrategia')
plt.legend(); plt.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

### ✅ Conclusión del Ejercicio 4

- **SMOTE** y **`class_weight='balanced'`** producen resultados similares en F1 y AUC.
- **`class_weight`** suele ser más rápido porque no aumenta el tamaño del train (evita la sobrecarga de sintetizar muestras).
- **SMOTE** puede ser más útil cuando el desbalance es extremo (>1:100) porque el modelo "ve" más variedad de casos de la minoritaria.
- En XGBoost usamos `scale_pos_weight = neg/pos` como equivalente a `class_weight='balanced'` en sklearn.
- **Ambas son alternativas válidas**; elige según velocidad, tamaño del dataset y sensibilidad a muestras sintéticas.


## 🏆 Síntesis final · Mejor configuración

Combinamos los aprendizajes de los 4 ejercicios para construir el "mejor" modelo posible y comparar contra baselines.


In [ ]:
# Configuración "campeona" según los ejercicios:
# - Usamos winsorización (ya aplicada)
# - max_depth moderado
# - Balanceo con class_weight (más rápido, resultado similar a SMOTE)
# - MLP con Dropout

modelo_campeon = {
    'RF ajustado': RandomForestClassifier(n_estimators=300, max_depth=8,
                                           class_weight='balanced',
                                           random_state=SEED, n_jobs=-1),
}

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
modelo_campeon['XGB ajustado'] = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=neg/pos,
    eval_metric='logloss', random_state=SEED, n_jobs=-1
)

# Baseline "malo": sin balanceo, max_depth alto
baseline_malo = RandomForestClassifier(n_estimators=100, max_depth=30, random_state=SEED, n_jobs=-1)

resultados_finales = []
for nombre, m in {'RF baseline (malo)': baseline_malo, **modelo_campeon}.items():
    m.fit(X_train, y_train)
    y_pred  = m.predict(X_test)
    y_proba = m.predict_proba(X_test)[:, 1]
    resultados_finales.append({
        'Modelo': nombre,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'AUC':       roc_auc_score(y_test, y_proba)
    })

df_final = pd.DataFrame(resultados_finales).set_index('Modelo')
df_final.round(4)

In [ ]:
# ROC comparativo final
plt.figure(figsize=(8, 6))
for nombre, m in {'RF baseline (malo)': baseline_malo, **modelo_campeon}.items():
    y_proba = m.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, linewidth=2.5, label=f'{nombre}  (AUC = {auc:.3f})')
plt.plot([0,1], [0,1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC: baseline vs modelos ajustados')
plt.legend(loc='lower right'); plt.grid(True); plt.tight_layout(); plt.show()

## 📋 Resumen de aprendizajes

| Ejercicio | Conclusión clave |
|---|---|
| 1. SMOTE on/off | Sin balanceo, el Recall de la clase minoritaria colapsa. Accuracy sola engaña. |
| 2. `max_depth` | Existe un valor óptimo intermedio; profundidad alta → sobreajuste. |
| 3. Regularización | Dropout y L2 reducen el gap train-val; son complementarios. |
| 4. `class_weight` | Alternativa eficiente a SMOTE, con resultados equivalentes en muchos casos. |

### 💭 Preguntas de discusión final

1. Si en tu proyecto el **Recall es crítico** (ej. detección de fraude, cáncer), ¿qué combinación de técnicas usarías?
2. **`class_weight`** no aumenta el tamaño del train, mientras que **SMOTE** sí. ¿Cuándo elegirías cada uno?
3. Con datasets muy grandes (millones de muestras), ¿el sobreajuste sigue siendo un problema? Justifica.
4. ¿Podrías combinar SMOTE + `class_weight` a la vez? ¿Sería recomendable?

### 🧪 Ejercicios adicionales para práctica

- Repite el Ejercicio 4 usando **ADASYN** en lugar de SMOTE. Compara resultados.
- Prueba **tuning de hiperparámetros** con `GridSearchCV` sobre el mejor modelo. ¿Cuánto puedes mejorar el AUC?
- Explora **umbrales de decisión distintos a 0.5** (usa `predict_proba` + threshold custom). ¿Puedes optimizar F1 subiendo o bajando el umbral?
